# Pandas Working with JSON 

In [27]:
import pandas as pd

data = [
    {
        "order_id": 101,
        "customer": {"name": "Anand", "city": "Indore"},
        "items": [
            {"product": "Laptop", "price": 50000},
            {"product": "Mouse", "price": 500}
        ],
        "payment": {"method": "UPI", "status": "Completed"}
    },
    {
        "order_id": 102,
        "customer": {"name": "Rahul", "city": "Delhi"},
        "items": [
            {"product": "Phone", "price": 20000}
        ],
        # "payment": {"method": "Card", "status": "Pending"}
    },
    {
        "order_id": 103,
        "customer": {"name": "Sneha", "city": "Mumbai"},
        "items": [
            {"product": "Tablet", "price": 15000},
            {"product": "Keyboard", "price": 1000}
        ],
        "payment": {"method": "COD", "status": "Completed"}
    }
]

- “Convert this raw JSON into a clean DataFrame where each row represents one product per order, and give insights.”

In [28]:
df = pd.json_normalize(data)

In [29]:
df

,order_id,items,customer.name,customer.city,payment.method,payment.status
0,101,"[{'product': 'Laptop', 'price': 50000}, {'prod...",Anand,Indore,UPI,Completed
1,102,"[{'product': 'Phone', 'price': 20000}]",Rahul,Delhi,NaN,NaN
2,103,"[{'product': 'Tablet', 'price': 15000}, {'prod...",Sneha,Mumbai,COD,Completed


In [31]:
df1 = pd.json_normalize(data , 'items' , ['order_id'  , ['customer' , 'name'] , ['customer' , 'city'] ,['payment' , 'method'] , ['payment' , 'status']] , errors='ignore')

In [32]:
df1

,product,price,order_id,customer.name,customer.city,payment.method,payment.status
0,Laptop,50000,101,Anand,Indore,UPI,Completed
1,Mouse,500,101,Anand,Indore,UPI,Completed
2,Phone,20000,102,Rahul,Delhi,NaN,NaN
3,Tablet,15000,103,Sneha,Mumbai,COD,Completed
4,Keyboard,1000,103,Sneha,Mumbai,COD,Completed


In [33]:
df1 = df1.rename(columns = {'customer.name' : 'customer_name' , 'customer.city' : 'customer_city' , 'payment.method' : 'payment_method' , 'payment.status' : 'payment_status'})

In [34]:
df1

,product,price,order_id,customer_name,customer_city,payment_method,payment_status
0,Laptop,50000,101,Anand,Indore,UPI,Completed
1,Mouse,500,101,Anand,Indore,UPI,Completed
2,Phone,20000,102,Rahul,Delhi,NaN,NaN
3,Tablet,15000,103,Sneha,Mumbai,COD,Completed
4,Keyboard,1000,103,Sneha,Mumbai,COD,Completed


In [12]:
ttl_rev = df1['price'].sum()

print(ttl_rev)

86500


In [14]:
rev_city = df1.groupby('customer_city')['price'].sum()

print(rev_city)

customer_city
Delhi     20000
Indore    50500
Mumbai    16000
Name: price, dtype: int64


In [26]:
df1['product'].value_counts()

# df1['product'].value_counts().idxmax()      # only product

product
Laptop      1
Mouse       1
Phone       1
Tablet      1
Keyboard    1
Name: count, dtype: int64

In [18]:
print(most_sold)

product
Keyboard    1
Laptop      1
Mouse       1
Phone       1
Tablet      1
Name: product, dtype: int64


In [23]:
cond = (df1['payment_status'] == 'Pending')

df1[cond]

,product,price,order_id,customer_name,customer_city,payment_method,payment_status
2,Phone,20000,102,Rahul,Delhi,Card,Pending


# Messy Dataset

In [35]:
import pandas as pd

data = [
    {
        "order_id": 201,
        "user": {"name": "Anand", "city": "Indore"},
        "restaurants": [
            {
                "name": "Dominos",
                "items": [
                    {"food": "Pizza", "price": 300},
                    {"food": "Garlic Bread", "price": 150}
                ]
            }
        ],
        "payment": {"method": "UPI", "status": "Completed"}
    },
    {
        "order_id": 202,
        "user": {"name": "Rahul", "city": "Delhi"},
        "restaurants": [],   # ❗ Empty list
        "payment": {"method": "Card", "status": "Failed"}
    },
    {
        "order_id": 203,
        "user": {"name": "Sneha"},   # ❗ Missing city
        "restaurants": [
            {
                "name": "KFC",
                "items": [
                    {"food": "Burger", "price": 200}
                ]
            }
        ]
        # ❗ Missing payment
    },
    {
        "order_id": 204,
        "user": {"name": "Amit", "city": "Pune"}
        # ❗ Missing restaurants completely
    }
]

In [36]:
df = pd.json_normalize(data)

In [37]:
df

,order_id,restaurants,user.name,user.city,payment.method,payment.status
0,201,"[{'name': 'Dominos', 'items': [{'food': 'Pizza...",Anand,Indore,UPI,Completed
1,202,[],Rahul,Delhi,Card,Failed
2,203,"[{'name': 'KFC', 'items': [{'food': 'Burger', ...",Sneha,NaN,NaN,NaN
3,204,NaN,Amit,Pune,NaN,NaN


- “Give me a clean dataset where each row is one food item ordered, along with user and payment details.”

## Understanding some attribute of json_normalize

- 👉 record_path = the list you want to convert into rows

- 👉 meta = extra columns you want to carry along

In [105]:
for d in data:
    if 'restaurants' not in d:
        d['restaurants'] = []
    if 'payment' not in d:
        d['payment'] = {}

df = pd.json_normalize(data , record_path='restaurants' , meta=[
        'order_id',
        ['user', 'name'],
        ['user', 'city'],
        ['payment', 'method'],
        ['payment', 'status']
    ], sep='_' , errors='ignore')



In [106]:
df

,order_id,user_name,user_city,payment_method,payment_status


# IPL Dummy DataSet

In [106]:
import pandas as pd

data = [
    {
        "match_id": 101,
        "match_info": {"team1": "MI", "team2": "CSK", "city": "Mumbai"},
        "innings": [
            {
                "team": "MI",
                "players": [
                    {"name": "Rohit Sharma", "runs": 45},
                    {"name": "Ishan Kishan", "runs": 30}
                ]
            }
        ],
        "result": {"winner": "MI", "margin": "20 runs"}
    },
    {
        "match_id": 102,
        "match_info": {"team1": "RCB", "team2": "KKR"},
        "innings": [],   # ❗ Empty innings
        "result": {"winner": "KKR", "margin": "5 wickets"}
    },
    {
        "match_id": 103,
        "match_info": {"team1": "DC"},   # ❗ Missing team2
        "innings": [
            {
                "team": "DC",
                "players": [
                    {"name": "Pant", "runs": 60}
                ]
            }
        ]
        # ❗ Missing result
    },
    {
        "match_id": 104,
        "match_info": {"team1": "RR", "team2": "SRH", "city": "Jaipur"}
        # ❗ Missing innings completely
    }
]

In [124]:
df = pd.json_normalize(data , sep='_')
df

df0 = df.explode('innings')
df0


for d in data:
    if 'innings' not in d:
        d['innings'] = []

df1 = pd.json_normalize(data , record_path='innings',  meta=['match_id', ['match_info', 'team1'],
                                                             ['match_info', 'team2'] , ['match_info', 'city'] , 
                                                             ['result' , 'winner'] , ['result' , 'margin']] ,sep='_', errors='ignore')
df1



df2 = df1.explode('players')
df2

df3 = pd.json_normalize(data , record_path=['innings' , 'players'],  meta=['match_id', ['match_info', 'team1'],
                                                             ['match_info', 'team2'] , ['match_info', 'city'] , 
                                                             ['result' , 'winner'] , ['result' , 'margin'] ] ,sep='_', errors='ignore')


df3

dff = pd.concat([df1,df2,df3])

dff

dff.drop(columns = ['players'])

dff = dff.fillna({
    "match_info_team2": "Unknown",
    "match_info_city": "Unknown",
    "result_winner": "No Result",
    "result_margin": "No Data",
    "inning_team": "Not Defined",
    "player_name": "No Player",
    "player_runs": 0
})

dff

,team,players,match_id,match_info_team1,match_info_team2,match_info_city,result_winner,result_margin,name,runs
0,MI,"[{'name': 'Rohit Sharma', 'runs': 45}, {'name'...",101,MI,CSK,Mumbai,MI,20 runs,NaN,NaN
1,DC,"[{'name': 'Pant', 'runs': 60}]",103,DC,Unknown,Unknown,No Result,No Data,NaN,NaN
0,MI,"{'name': 'Rohit Sharma', 'runs': 45}",101,MI,CSK,Mumbai,MI,20 runs,NaN,NaN
0,MI,"{'name': 'Ishan Kishan', 'runs': 30}",101,MI,CSK,Mumbai,MI,20 runs,NaN,NaN
1,DC,"{'name': 'Pant', 'runs': 60}",103,DC,Unknown,Unknown,No Result,No Data,NaN,NaN
0,NaN,NaN,101,NaN,Unknown,Unknown,No Result,No Data,Rohit Sharma,45.0
1,NaN,NaN,101,NaN,Unknown,Unknown,No Result,No Data,Ishan Kishan,30.0
2,NaN,NaN,103,NaN,Unknown,Unknown,No Result,No Data,Pant,60.0


In [126]:
# STEP 1: Ensure 'innings' exists in all records
for d in data:
    if 'innings' not in d:
        d['innings'] = []

# STEP 2: Normalize innings level
df1 = pd.json_normalize(
    data,
    record_path='innings',
    meta=[
        'match_id',
        ['match_info', 'team1'],
        ['match_info', 'team2'],
        ['match_info', 'city'],
        ['result', 'winner'],
        ['result', 'margin']
    ],
    sep='_',
    errors='ignore'
)

# STEP 3: Handle players column (if exists)
if 'players' in df1.columns:

    # explode players list
    df2 = df1.explode('players')

    # normalize player data
    df3 = pd.json_normalize(df2['players'])

    # 🔥 IMPORTANT FIX (index issue)
    df2 = df2.reset_index(drop=True)
    df3 = df3.reset_index(drop=True)

    # merge properly
    df_final = pd.concat(
        [df2.drop(columns=['players']), df3],
        axis=1
    )

else:
    df_final = df1.copy()

# STEP 4: Fill missing values
df_final = df_final.fillna({
    "match_info_team2": "Unknown",
    "match_info_city": "Unknown",
    "result_winner": "No Result",
    "result_margin": "No Data",
    "inning_team": "Not Defined",
    "player_name": "No Player",
    "player_runs": 0
})

# FINAL OUTPUT
df_final

,team,match_id,match_info_team1,match_info_team2,match_info_city,result_winner,result_margin,name,runs
0,MI,101,MI,CSK,Mumbai,MI,20 runs,Rohit Sharma,45
1,MI,101,MI,CSK,Mumbai,MI,20 runs,Ishan Kishan,30
2,DC,103,DC,Unknown,Unknown,No Result,No Data,Pant,60


In [121]:
import pandas as pd
import numpy as np

# 1. Normalize the root level
df = pd.json_normalize(data, sep='_')

# 2. Explode the innings (This safely keeps empty matches like 102 and 104)
df = df.explode('innings').reset_index(drop=True)

# 3. Safely extract 'team' and 'players' from the dictionary
def extract_innings(row):
    if isinstance(row['innings'], dict):
        return pd.Series([row['innings'].get('team'), row['innings'].get('players')])
    return pd.Series([np.nan, []])

df[['inning_team', 'players']] = df.apply(extract_innings, axis=1)

# 4. Explode the players
df = df.explode('players').reset_index(drop=True)

# 5. Safely extract 'name' and 'runs' from the dictionary
def extract_players(row):
    if isinstance(row['players'], dict):
        return pd.Series([row['players'].get('name'), row['players'].get('runs')])
    return pd.Series([np.nan, np.nan])

df[['player_name', 'player_runs']] = df.apply(extract_players, axis=1)

# 6. Drop the dirty columns and fill missing data
final_df = df.drop(columns=['innings', 'players'])

final_df = final_df.fillna({
    "match_info_team2": "Unknown",
    "match_info_city": "Unknown",
    "result_winner": "No Result",
    "result_margin": "No Data",
    "inning_team": "Not Defined",
    "player_name": "No Player",
    "player_runs": 0
})

final_df

,match_id,match_info_team1,match_info_team2,match_info_city,result_winner,result_margin,inning_team,player_name,player_runs
0,101,MI,CSK,Mumbai,MI,20 runs,MI,Rohit Sharma,45.0
1,101,MI,CSK,Mumbai,MI,20 runs,MI,Ishan Kishan,30.0
2,102,RCB,KKR,Unknown,KKR,5 wickets,Not Defined,No Player,0.0
3,103,DC,Unknown,Unknown,No Result,No Data,DC,Pant,60.0
4,104,RR,SRH,Jaipur,No Result,No Data,Not Defined,No Player,0.0


In [88]:
df = pd.json_normalize(data , sep='_')
df

df0 = df.explode('innings')
df0


df1 = pd.json_normalize(data , record_path='innings' , sep='_')
df1

df1 = df1.explode('players')

df1

df1 = df1.drop(columns= 'players')
df = df.drop(columns= 'innings')

In [33]:
for d in data:
    if 'innings' not in d:
        d['innings'] = []


df = pd.json_normalize(data , record_path=['innings' , 'players'] ,  meta=['match_id', ['match_info', 'team1'], ['match_info', 'team2'] , ['match_info', 'city'] , ['innings' , 'team'] , ['result' , 'winner'] , ['result' , 'margin']] ,sep='_', errors='ignore')

df = df.fillna({
    "match_info_team2": "Unknown",
    "match_info_city": "Unknown",
    "result_winner": "No Result",
    "result_margin": "No Data",
    'innings' : 'No Defined'
})
df

# print(df_norm_2.isna().sum())

,name,runs,match_id,match_info_team1,match_info_team2,match_info_city,innings_team,result_winner,result_margin
0,Rohit Sharma,45,101,NaN,Unknown,Unknown,MI,No Result,No Data
1,Ishan Kishan,30,101,NaN,Unknown,Unknown,MI,No Result,No Data
2,Pant,60,103,NaN,Unknown,Unknown,DC,No Result,No Data


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   name              3 non-null      object
 1   runs              3 non-null      int64 
 2   match_id          3 non-null      object
 3   match_info.team1  0 non-null      object
 4   match_info.team2  0 non-null      object
 5   match_info.city   0 non-null      object
 6   innings.team      3 non-null      object
 7   result.winner     0 non-null      object
 8   result.margin     0 non-null      object
dtypes: int64(1), object(8)
memory usage: 348.0+ bytes


In [14]:
obj_cols = df.select_dtypes(include='object').columns
df[obj_cols] = df[obj_cols].fillna("Unknown")

C:\Users\Anand Patidar\AppData\Local\Temp\ipykernel_32976\205281982.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[obj_cols] = df[obj_cols].fillna("Unknown")


In [15]:
df

,name,runs,match_id,match_info.team1,match_info.team2,match_info.city,innings.team,result.winner,result.margin
0,Rohit Sharma,45,101,Unknown,Unknown,Unknown,MI,Unknown,Unknown
1,Ishan Kishan,30,101,Unknown,Unknown,Unknown,MI,Unknown,Unknown
2,Pant,60,103,Unknown,Unknown,Unknown,DC,Unknown,Unknown


In [168]:
print(df.isna().sum())

name                0
runs                0
match_id            0
match_info.team1    0
match_info.team2    0
match_info.city     0
innings.team        0
result.winner       0
result.margin       0
dtype: int64


In [24]:
import pandas as pd

data = [
    {
        "match_id": 101,
        "match_info": {"team1": "MI", "team2": "CSK", "city": "Mumbai"},
        "innings": [
            {
                "team": "MI",
                "players": [
                    {"name": "Rohit Sharma", "runs": 45},
                    {"name": "Ishan Kishan", "runs": 30}
                ]
            }
        ],
        "result": {"winner": "MI", "margin": "20 runs"}
    },
    {
        "match_id": 102,
        "match_info": {"team1": "RCB", "team2": "KKR"},
        "innings": [],   # ❗ Empty innings
        "result": {"winner": "KKR", "margin": "5 wickets"}
    },
    {
        "match_id": 103,
        "match_info": {"team1": "DC"},   # ❗ Missing team2
        "innings": [
            {
                "team": "DC",
                "players": [
                    {"name": "Pant", "runs": 60}
                ]
            }
        ]
        # ❗ Missing result
    },
    {
        "match_id": 104,
        "match_info": {"team1": "RR", "team2": "SRH", "city": "Jaipur"}
        # ❗ Missing innings completely
    }
]

In [25]:
df1 = pd.json_normalize(data , sep='_')

df1

,match_id,innings,match_info_team1,match_info_team2,match_info_city,result_winner,result_margin
0,101,"[{'team': 'MI', 'players': [{'name': 'Rohit Sh...",MI,CSK,Mumbai,MI,20 runs
1,102,[],RCB,KKR,NaN,KKR,5 wickets
2,103,"[{'team': 'DC', 'players': [{'name': 'Pant', '...",DC,NaN,NaN,NaN,NaN
3,104,NaN,RR,SRH,Jaipur,NaN,NaN


In [26]:
df2 = df1.explode('innings')
df2

,match_id,innings,match_info_team1,match_info_team2,match_info_city,result_winner,result_margin
0,101,"{'team': 'MI', 'players': [{'name': 'Rohit Sha...",MI,CSK,Mumbai,MI,20 runs
1,102,NaN,RCB,KKR,NaN,KKR,5 wickets
2,103,"{'team': 'DC', 'players': [{'name': 'Pant', 'r...",DC,NaN,NaN,NaN,NaN
3,104,NaN,RR,SRH,Jaipur,NaN,NaN


In [27]:
inning_extract = df2['innings'].dropna().apply(pd.Series)

inning_extract

,team,players
0,MI,"[{'name': 'Rohit Sharma', 'runs': 45}, {'name'..."
2,DC,"[{'name': 'Pant', 'runs': 60}]"


In [28]:
df3 = inning_extract.explode('players')
df3

,team,players
0,MI,"{'name': 'Rohit Sharma', 'runs': 45}"
0,MI,"{'name': 'Ishan Kishan', 'runs': 30}"
2,DC,"{'name': 'Pant', 'runs': 60}"


In [29]:
player_detail = df3['players'].dropna().apply(pd.Series)

player_detail

,name,runs
0,Rohit Sharma,45
0,Ishan Kishan,30
2,Pant,60


In [35]:
df4 = df3.drop(columns=['players']).join(player_detail).drop_duplicates()

df4

,team,name,runs
0,MI,Rohit Sharma,45
0,MI,Ishan Kishan,30
2,DC,Pant,60


In [36]:
df4 = df4.rename(columns={'name' : 'player_name' , 'runs' : 'player_runs'})

df4.head()

,team,player_name,player_runs
0,MI,Rohit Sharma,45
0,MI,Ishan Kishan,30
2,DC,Pant,60


In [37]:
final = df1.drop(columns=['innings']).join(df4)

final

,match_id,match_info_team1,match_info_team2,match_info_city,result_winner,result_margin,team,player_name,player_runs
0,101,MI,CSK,Mumbai,MI,20 runs,MI,Rohit Sharma,45.0
0,101,MI,CSK,Mumbai,MI,20 runs,MI,Ishan Kishan,30.0
1,102,RCB,KKR,NaN,KKR,5 wickets,NaN,NaN,NaN
2,103,DC,NaN,NaN,NaN,NaN,DC,Pant,60.0
3,104,RR,SRH,Jaipur,NaN,NaN,NaN,NaN,NaN


In [33]:
final = final.drop_duplicates()

final

,match_id,match_info_team1,match_info_team2,match_info_city,result_winner,result_margin,team,player_name,player_runs
0,101,MI,CSK,Mumbai,MI,20 runs,MI,Rohit Sharma,45.0
0,101,MI,CSK,Mumbai,MI,20 runs,MI,Ishan Kishan,30.0
1,102,RCB,KKR,NaN,KKR,5 wickets,NaN,NaN,NaN
2,103,DC,NaN,NaN,NaN,NaN,DC,Pant,60.0
3,104,RR,SRH,Jaipur,NaN,NaN,NaN,NaN,NaN


In [34]:
final = final.fillna('Not available')
final

,match_id,match_info_team1,match_info_team2,match_info_city,result_winner,result_margin,team,player_name,player_runs
0,101,MI,CSK,Mumbai,MI,20 runs,MI,Rohit Sharma,45.0
0,101,MI,CSK,Mumbai,MI,20 runs,MI,Ishan Kishan,30.0
1,102,RCB,KKR,Not available,KKR,5 wickets,Not available,Not available,Not available
2,103,DC,Not available,Not available,Not available,Not available,DC,Pant,60.0
3,104,RR,SRH,Jaipur,Not available,Not available,Not available,Not available,Not available
